In [1]:
import os
import sys
from pathlib import Path
from typing import cast

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.getcwd())

import torch
from omegaconf import DictConfig, OmegaConf

from core.data.mining import load_cache
from core.data.module import AlignData
from core.data.vocab import TagTokenizer
from core.model.bobert import BobertForAlignment
from core.training.align import find_pretraining_checkpoint, load_pretraining_weights, setup_alignment, train
from core.training.setup import setup_device

In [2]:
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {setup_device()}")
print(f"Working directory: {os.getcwd()}")

config = cast(DictConfig, OmegaConf.load("./config.yaml"))
config.alignment.checkpoint_dir = "./experiments/alignment"
print(OmegaConf.to_yaml(config))

PyTorch version: 2.9.0+cu126
Using device: gpu
Working directory: /home/jessiez/projects/bobert
data:
  dataset_path: ./data/dataset
  dataset_seed: 42
  max_seq_len: 2048
  val_split: 0.1
  min_sr: 2.0
  max_sr: 14.0
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward: 1280
  dropout: 0.1
  local_attention_window: 128
  global_attention_layers:
  - 2
  - 5
components:
  compile_model: true
  compile_mode: default
  activation_checkpointing: true
pretraining:
  pretrain_size: null
  batch_size: 32
  num_epochs: 8
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 4
  muon_lr: 0.02
  muon_wd: 0.01
  adam_lr: 0.0002
  adam_wd: 0.05
  adam_betas:
  - 0.9
  - 0.95
  min_lr: 1.0e-06
  cooldown_type: cosine
  warmup_ratio: 0.1
  stable_ratio: 0.1
  use_amp: true
  checkpoint_dir: ./experiments
  difficulty_loss_weight: 1.0
  mlm_loss_weight: 1.0
  pooling_stats:
  - mean
  - max
  - std
  pooling_stat_dim: 256
  masking_ratio: 0.25
  mean_span_length: 4
  sampling:
    kde_b

In [3]:
cache_path = Path(config.alignment.mining_cache_path)
cache = load_cache(
    cache_path,
    alignment_size=config.alignment.get("alignment_size", None),
    random_seed=config.alignment.get("mining_cache_seed", 42),
)

cache.head(), len(cache)

(shape: (5, 14)
 ┌────────────┬───────────┬──────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ beatmap_id ┆ status_gr ┆ stars    ┆ aim      ┆ … ┆ cross_sta ┆ hard_nega ┆ hard_nega ┆ lgcn_embe │
 │ ---        ┆ oup       ┆ ---      ┆ ---      ┆   ┆ tus_posit ┆ tive_ids  ┆ tive_weig ┆ dding     │
 │ i64        ┆ ---       ┆ f64      ┆ f64      ┆   ┆ ive_weigh ┆ ---       ┆ hts       ┆ ---       │
 │            ┆ str       ┆          ┆          ┆   ┆ ts        ┆ list[i64] ┆ ---       ┆ list[f64] │
 │            ┆           ┆          ┆          ┆   ┆ ---       ┆           ┆ list[f64] ┆           │
 │            ┆           ┆          ┆          ┆   ┆ list[f64] ┆           ┆           ┆           │
 ╞════════════╪═══════════╪══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ 97840      ┆ unranked  ┆ 6.312519 ┆ 2.857514 ┆ … ┆ [0.6572,  ┆ [812575,  ┆ [1.0,     ┆ [0.014923 │
 │            ┆           ┆          ┆          ┆   ┆ 0.649898, ┆ 

In [4]:
datamodule = AlignData(config, TagTokenizer())
datamodule.prepare_data()
datamodule.setup("fit")

Pre-filtered to load 98337 specific beatmap IDs.
Selected 92382 beatmaps. Processing in chunks of 5000...


Processing Chunks: 100%|██████████| 19/19 [01:16<00:00,  4.05s/it]


Loaded data for 92234 beatmaps.
Data split: 83011 training, 9223 validation


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")
model = BobertForAlignment.from_config(config, device)

summary = model.get_summary()
print(f"\n--- BERT Encoder Information ---")
print(f"Total Parameters: {summary['trainable_parameters'] / 1e6:.2f}M")
print(f"Model Dimension: {model.bert.d_model}")
print(f"Number of Heads: {model.bert.n_heads}")
print(f"Number of Layers: {model.bert.n_layers}")

/home/jessiez/projects/bobert/.venv/lib/python3.12/site-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


Compiling BERT alignment model with torch.compile...

--- BERT Encoder Information ---
Total Parameters: 6.03M
Model Dimension: 512
Number of Heads: 8
Number of Layers: 6


In [6]:
pretrain_checkpoint = find_pretraining_checkpoint(Path("experiments") / "checkpoints")
stats = load_pretraining_weights(model, pretrain_checkpoint)
print(
    f"loaded={stats['loaded']} skipped={stats['skipped']} "
    f"missing={stats['missing']} unexpected={stats['unexpected']}"
)
print(f"checkpoint={stats['checkpoint_path']}")
print(f"loaded difficulty head: {stats['loaded_difficulty_head']}")

loaded=51 skipped=27 missing=20 unexpected=0
checkpoint=experiments/checkpoints/last.ckpt
loaded difficulty head: True


In [7]:
module, trainer = setup_alignment(config, model, datamodule.normalizer)

print(f"\nAlignment setup complete.")
print(f"Total epochs: {config.alignment.num_epochs}")
print(f"Training samples: {len(datamodule.train_dataset)}")
print(f"Validation samples: {len(datamodule.val_dataset)}")

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores



Alignment setup complete.
Total epochs: 3
Training samples: 83011
Validation samples: 9223


In [8]:
train(module, trainer, datamodule)

print("\nBoBERT alignment completed!")

/home/jessiez/projects/bobert/.venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/jessiez/projects/bobert/experiments/alignment/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.


Optimizer initialized: 24 Muon params, 47 AdamW params.
Scheduler: WSD with 97 warmup, 0 stable, 1850 decay steps.
Cooldown type: linear, Min LR Ratio: 0.0333


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=3` reached.



BoBERT alignment completed!
